In [0]:
%python
dbutils.widgets.text("checkpoint_name", "ADHOC")
dbutils.widgets.text("catalog", "vp_dp20040222")

checkpoint_name = dbutils.widgets.get("checkpoint_name")
catalog = dbutils.widgets.get("catalog")

print("Checkpoint:", checkpoint_name)
print("Catalog:", catalog)

In [0]:
CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog || '.ops.reconciliation_results') (
    run_ts TIMESTAMP,
    check_name STRING,
    source_value DECIMAL(18,3),
    target_value DECIMAL(18,3),
    status STRING
) USING DELTA;


In [0]:
INSERT INTO IDENTIFIER(:catalog || '.ops.reconciliation_results')
    (run_ts, check_name, source_value, target_value, status)

WITH reconciliation AS
(
    -- 1. SESSION_COUNT
    SELECT
        'SESSION_COUNT' AS check_name,
        CAST(
            COUNT(DISTINCT UPPER(TRIM(session_id)))
            AS DECIMAL(18,3)
        ) AS source_value,
        CAST(
            (
                (SELECT COUNT(*) FROM IDENTIFIER(:catalog || '.silver.sessions'))
                +
                (SELECT COUNT(*) FROM IDENTIFIER(:catalog || '.silver.quarantine_sessions'))
            )
            AS DECIMAL(18,3)
        ) AS target_value
    FROM IDENTIFIER(:catalog || '.bronze.sessions')

    UNION ALL

    -- 2. ENERGY_KWH
    SELECT
        'ENERGY_KWH',
        CAST(
            COALESCE(SUM(energy_kwh), 0)
            AS DECIMAL(18,3)
        ),
        CAST(
            COALESCE((SELECT SUM(energy_kwh) FROM IDENTIFIER(:catalog || '.gold.agg_daily_station')), 0)
            AS DECIMAL(18,3)
        )
    FROM IDENTIFIER(:catalog || '.silver.sessions')
    WHERE UPPER(TRIM(status)) = 'COMPLETED'

    UNION ALL

    -- 3. CURRENT_MEMBERS
    SELECT
        'CURRENT_MEMBERS',
        CAST(
            (SELECT COUNT(DISTINCT member_id) FROM IDENTIFIER(:catalog || '.silver.members'))
            AS DECIMAL(18,3)
        ),
        CAST(
            (SELECT COUNT(*) FROM IDENTIFIER(:catalog || '.gold.dim_member') WHERE is_current = TRUE)
            AS DECIMAL(18,3)
        )
)

SELECT
    CURRENT_TIMESTAMP() AS run_ts,
    check_name,
    source_value,
    target_value,
    CASE
        WHEN source_value = target_value THEN 'PASS'
        ELSE 'FAIL'
    END AS status
FROM reconciliation;

In [0]:
SELECT *
FROM vp_dp20040222.ops.reconciliation_results
ORDER BY run_ts DESC, check_name;

In [0]:
%python
from pyspark.sql import functions as F

# H2 — Checkpoint counts

# The tables tracked across bronze, silver, and gold layers
tables_to_count = [
    f"{catalog}.bronze.sessions",
    f"{catalog}.bronze.members",
    f"{catalog}.bronze.chargers",
    f"{catalog}.bronze.stations",
    f"{catalog}.bronze.tariffs",

    f"{catalog}.silver.sessions",
    f"{catalog}.silver.quarantine_sessions",
    f"{catalog}.silver.members",
    f"{catalog}.silver.chargers",
    f"{catalog}.silver.stations",
    f"{catalog}.silver.tariffs",
    f"{catalog}.silver.session_faults",

    f"{catalog}.gold.fact_sessions",
    f"{catalog}.gold.agg_daily_station",
]

# Count each table and build one record per table
checkpoint_rows = []

for table_name in tables_to_count:
    row_count = spark.read.table(table_name).count()

    checkpoint_rows.append((
        checkpoint_name,
        table_name,
        int(row_count)
    ))

    print(f"{table_name}: {row_count}")

# Create the checkpoint DataFrame with the required schema
checkpoint_df = (
    spark.createDataFrame(
        checkpoint_rows,
        schema=[
            "checkpoint_name",
            "table_name",
            "row_count"
        ]
    )
    .withColumn("row_count", F.col("row_count").cast("long"))
    .withColumn("captured_at", F.current_timestamp())
    .select(
        "checkpoint_name",
        "table_name",
        "row_count",
        "captured_at"
    )
)

# Create the destination table on the first run
checkpoint_table = f"{catalog}.ops.checkpoint_counts"

if not spark.catalog.tableExists(checkpoint_table):
    (
        checkpoint_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(checkpoint_table)
    )
else:
    # Append a new snapshot; do not overwrite previous checkpoints.
    (
        checkpoint_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(checkpoint_table)
    )

print(f"Checkpoint saved: {checkpoint_name}")

In [0]:
SELECT
    *
FROM vp_dp20040222.ops.checkpoint_counts
ORDER BY captured_at DESC, table_name;